# 03 — Feature Engineering: Historical & Agronomic Features
**Crop Yield Prediction and Forecasting Using Agricultural Data and Machine Learning**  
*Author: Aryan Puranik*

This notebook engineers historical yield lags, rolling temporal statistics, and climatic anomaly features from the historical crop records.

### Objectives & Research Rationale
1. **Historical Yield Lags:** Does past yield of the *same crop in the same district* improve one-year-ahead forecasts?
2. **Leakage-Safe Temporal Features:** Lags ($t-1, t-2, t-3, t-5$) are constructed strictly using past calendar years.
3. **Outlier-Sanitized History:** Erroneous data-entry typos (e.g., Chickpea 2007 entries > 10,000 kg/ha) are masked during lag generation so they do not contaminate subsequent years.
4. **Benchmark Ladder:** Establish naive baselines (Previous-Year, Rolling 3-Year Mean, and Trend-Adjusted Naive) on the 2012–2014 validation period.
5. **Feature Handoff:** Export full engineered sets (`train_engineered.csv`, `val_engineered.csv`, `test_engineered.csv`) and `feature_sets.json` for subsequent feature selection and model notebooks.


## 1. Setup, Imports, and Data Loading


In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "crop_yield_dataset.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "feature_engineering"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

assert RAW_PATH.is_file(), f"Dataset not found: {RAW_PATH}"

raw_df = pd.read_csv(RAW_PATH)
print("Raw Dataset Dimensions:", raw_df.shape)
raw_df.head(3)


Raw Dataset Dimensions: (198369, 15)


## 2. Key Verification and Outlier Sanitization for Historical Features

`Region + Crop + Year` uniquely identifies an observation.
To prevent known data entry errors (e.g., 2007 Chickpea values of 58,363 kg/ha and 32,000 kg/ha flagged in Phase 1 audit) from propagating into subsequent lag features (`Yield_Lag1` in 2008, `Yield_Rolling3` in 2008–2010, and multi-year changes in 2012), we mask these extreme values as `NaN` in the historical lookup series.


In [4]:
KEY = ["Region", "Crop", "Year"]
assert set(KEY + ["Yield"]).issubset(raw_df.columns)
assert not raw_df.duplicated(KEY).any(), "Duplicate keys found!"

# Sanitize known data-entry typos in the history lookup series
clean_history_df = raw_df.copy()
chickpea_outliers = (
    (clean_history_df["Crop"] == "Chickpea") &
    (clean_history_df["Year"] == 2007) &
    (clean_history_df["Yield"] > 10000)
)
print(f"Masked {chickpea_outliers.sum()} anomalous Chickpea 2007 records from historical lag propagation.")
clean_history_df.loc[chickpea_outliers, "Yield"] = np.nan


Masked 2 anomalous Chickpea 2007 records from historical lag propagation.


## 3. Construct Calendar-Year Lag and Rolling Temporal Features

We join on exact calendar years (`Year - lag`) rather than arbitrary row shifts, ensuring that missing intermediate years correctly produce `NaN` values rather than misleadingly shifting data across years.

Engineered features:
- **`Yield_Lag1`, `Yield_Lag2`, `Yield_Lag3`, `Yield_Lag5`**: Previous yields.
- **`Yield_Rolling3`**: Mean of previous 3 years ($t-1, t-2, t-3$).
- **`Yield_Rolling3_Std`**: Standard deviation over previous 3 years (yield volatility).
- **`Yield_Change_1to5`**: Change from 5 years ago to 1 year ago ($Y_{t-1} - Y_{t-5}$).
- **`Yield_Trend_Annual`**: Estimated annual trend rate $rac{Y_{t-1} - Y_{t-5}}{4}$.
- **`Yield_Trend_Naive`**: Trend-adjusted naive estimate $Y_{t-1} + rac{Y_{t-1} - Y_{t-3}}{2}$.
- **`Temperature_Anomaly`, `Rainfall_Anomaly`**: Deviation from expanding historical mean.


In [6]:
def engineer_features(data, history_source):
    result = data.copy()
    history = history_source[["Region", "Crop", "Year", "Yield"]].copy()

    # Exact calendar-year lags
    for lag in (1, 2, 3, 5):
        prev = history.rename(columns={
            "Year": "Hist_Year",
            "Yield": f"Yield_Lag{lag}"
        })
        result["Hist_Year"] = result["Year"] - lag
        result = result.merge(
            prev,
            on=["Region", "Crop", "Hist_Year"],
            how="left",
            validate="many_to_one"
        ).drop(columns="Hist_Year")

    # 3-year rolling statistics
    lag_cols = ["Yield_Lag1", "Yield_Lag2", "Yield_Lag3"]
    result["Yield_Rolling3"] = result[lag_cols].mean(axis=1, skipna=False)
    result["Yield_Rolling3_Std"] = result[lag_cols].std(axis=1, ddof=0, skipna=False)

    # 5-year change and annual trend
    result["Yield_Change_1to5"] = result["Yield_Lag1"] - result["Yield_Lag5"]
    result["Yield_Trend_Annual"] = (result["Yield_Lag1"] - result["Yield_Lag5"]) / 4.0
    result["Yield_Trend5"] = result["Yield_Change_1to5"]  # backward-compatible alias

    # Trend-adjusted naive estimate: Lag1 + average 2-year trend
    result["Yield_Trend_Naive"] = result["Yield_Lag1"] + (result["Yield_Lag1"] - result["Yield_Lag3"]) / 2.0

    # Availability indicators
    result["History_Lag1_Available"] = result["Yield_Lag1"].notna().astype(int)
    result["History_3yr_Complete"] = result[lag_cols].notna().all(axis=1).astype(int)

    # Climate anomalies (expanding historical mean up to year t-1)
    climate = data[["Region", "Crop", "Year", "Temperature", "Rainfall"]].sort_values(["Region", "Crop", "Year"]).copy()
    climate["Temp_shift"] = climate.groupby(["Region", "Crop"])["Temperature"].shift(1)
    climate["Rain_shift"] = climate.groupby(["Region", "Crop"])["Rainfall"].shift(1)
    climate["Temperature_Anomaly"] = climate["Temperature"] - climate.groupby(["Region", "Crop"])["Temp_shift"].expanding(min_periods=3).mean().values
    climate["Rainfall_Anomaly"] = climate["Rainfall"] - climate.groupby(["Region", "Crop"])["Rain_shift"].expanding(min_periods=3).mean().values

    result = result.merge(
        climate[["Region", "Crop", "Year", "Temperature_Anomaly", "Rainfall_Anomaly"]],
        on=["Region", "Crop", "Year"],
        how="left",
        validate="one_to_one"
    )

    return result

data_fe = engineer_features(raw_df, clean_history_df)
print("Engineered DataFrame shape:", data_fe.shape)
data_fe[["Year", "Region", "Crop", "Yield", "Yield_Lag1", "Yield_Rolling3", "Yield_Change_1to5", "Temperature_Anomaly"]].tail(5)


Engineered DataFrame shape: (198369, 29)


## 4. History Availability Across Chronological Partitions

We inspect the proportion of observations where 1-year lag and complete 3-year history are available. Missing lags primarily occur in early survey years (1966–1970) or when a district-crop series began intermittently.


In [8]:
data_fe["Partition"] = np.select(
    [data_fe["Year"] <= 2011, data_fe["Year"].between(2012, 2014), data_fe["Year"] >= 2015],
    ["Train (<=2011)", "Validation (2012-2014)", "Test (2015-2017)"],
    default="Other"
)

coverage = data_fe.groupby("Partition")[["History_Lag1_Available", "History_3yr_Complete"]].mean() * 100
display(coverage.round(2))

fig, ax = plt.subplots(figsize=(7, 4))
coverage.plot(kind="bar", ax=ax, rot=0, colormap="viridis")
ax.set_title("Historical Feature Availability by Partition")
ax.set_ylabel("Availability (%)")
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()


                        History_Lag1_Available  History_3yr_Complete
Partition                                                           
Test (2015-2017)                         93.83                 84.25
Train (<=2011)                           93.60                 84.63
Validation (2012-2014)                   93.52                 87.84


## 5. Baseline Evaluation on Validation Cohorts (2012–2014)

We evaluate simple heuristic baselines on:
1. **Complete-History Cohort ($N=9,966$):** Rows where all past 3 years are available.
2. **Full Validation Cohort ($N=11,345$):** All validation rows, using district-crop training median as fallback when history is missing.

This establishes the baseline benchmark ladder for subsequent machine learning models.


In [10]:
# Compute training historical medians (<= 2011) for full-cohort fallback
train_mask = data_fe["Year"] <= 2011
val_mask = data_fe["Year"].between(2012, 2014)

train_data = data_fe[train_mask]
val_fe = data_fe[val_mask].copy()

rc_median = train_data.groupby(["Region", "Crop"])["Yield"].median().rename("Hist_Median")
crop_median = train_data.groupby("Crop")["Yield"].median().rename("Crop_Median")

val_fe = val_fe.merge(rc_median, on=["Region", "Crop"], how="left")
val_fe = val_fe.merge(crop_median, on="Crop", how="left")
val_fe["Hist_Median"] = val_fe["Hist_Median"].fillna(val_fe["Crop_Median"])
val_fe["Fallback_Rolling3"] = val_fe["Yield_Rolling3"].fillna(val_fe["Hist_Median"])

# Evaluation function
def eval_baseline(actual, pred, label, n_rows):
    err = actual - pred
    mae = np.mean(np.abs(err))
    rmse = np.sqrt(np.mean(err**2))
    r2 = 1.0 - np.sum(err**2) / np.sum((actual - np.mean(actual))**2)
    valid_map = actual >= 10.0
    map_err = 100.0 * np.mean(np.abs(err[valid_map] / actual[valid_map]))
    return {
        "Cohort": label,
        "Rows": n_rows,
        "MAE (kg/ha)": mae,
        "RMSE (kg/ha)": rmse,
        "R2": r2,
        "MAP (%)": map_err
    }

results = []
valid_complete = val_fe[val_fe["History_3yr_Complete"] == 1]
y_comp = valid_complete["Yield"].values

results.append(eval_baseline(y_comp, valid_complete["Yield_Lag1"].values, "Complete History: Prev Year (Lag 1)", len(valid_complete)))
results.append(eval_baseline(y_comp, valid_complete["Yield_Rolling3"].values, "Complete History: 3-Year Mean", len(valid_complete)))
results.append(eval_baseline(y_comp, valid_complete["Yield_Trend_Naive"].values, "Complete History: Trend-Adjusted Naive", len(valid_complete)))
results.append(eval_baseline(val_fe["Yield"].values, val_fe["Fallback_Rolling3"].values, "Full Validation: 3-Year Mean (with Fallback)", len(val_fe)))

baseline_df = pd.DataFrame(results)
display(baseline_df.round(2))


                                         Cohort   Rows  ...    R2  MAP (%)
0           Complete History: Prev Year (Lag 1)   9966  ...  0.88    29.90
1                 Complete History: 3-Year Mean   9966  ...  0.89    29.39
2        Complete History: Trend-Adjusted Naive   9966  ...  0.81    39.11
3  Full Validation: 3-Year Mean (with Fallback)  11345  ...  0.86    30.67

[4 rows x 6 columns]


## 6. Formal Feature Set Configurations

We structure candidate predictor subsets to systematically evaluate agronomic features vs. engineered historical signals in Notebook 04 and model notebooks:
- **`E0_climate`**: Climate + climate anomalies
- **`E1_climate_soil`**: Climate + soil & primary nutrients (pH, N, P, K)
- **`E2_traditional_ag`**: Full traditional set including Fertilizer and Pesticide
- **`E3_engineered_full`**: Traditional features + Historical yield lags & availability flags


In [12]:
CORE = ["Region", "Crop", "Year"]
WEATHER = ["Temperature", "Rainfall", "Humidity", "Temperature_Anomaly", "Rainfall_Anomaly"]
SOIL_NUTRIENTS = ["Soil Type", "pH", "Nitrogen", "Phosphorus", "Potassium"]
AGROCHEMICALS = ["Fertilizer", "Pesticide"]
HISTORY = [
    "Yield_Lag1", "Yield_Lag2", "Yield_Lag3", "Yield_Lag5",
    "Yield_Rolling3", "Yield_Rolling3_Std", "Yield_Change_1to5",
    "History_Lag1_Available", "History_3yr_Complete"
]

feature_sets = {
    "E0_climate": CORE + WEATHER,
    "E1_climate_soil": CORE + WEATHER + SOIL_NUTRIENTS,
    "E2_traditional_ag": CORE + WEATHER + SOIL_NUTRIENTS + AGROCHEMICALS,
    "E3_engineered_full": CORE + WEATHER + SOIL_NUTRIENTS + AGROCHEMICALS + HISTORY
}

for name, cols in feature_sets.items():
    print(f"{name:20s}: {len(cols)} features")


E0_climate          : 8 features
E1_climate_soil     : 13 features
E2_traditional_ag   : 15 features
E3_engineered_full  : 24 features


## 7. Export Processed Datasets and Feature Metadata

We export:
1. `train_engineered.csv`, `val_engineered.csv`, and `test_engineered.csv` into `data/processed/`.
2. `feature_sets.json` containing standardized feature groupings.
3. Modular exports in `data/processed/feature_engineering/` for reference.


In [14]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Merge engineered columns with existing processed partitions to preserve approved preprocessing
history_export_cols = [
    "Yield_Lag1", "Yield_Lag2", "Yield_Lag3", "Yield_Lag5",
    "Yield_Rolling3", "Yield_Rolling3_Std", "Yield_Change_1to5", "Yield_Trend5",
    "History_Lag1_Available", "History_3yr_Complete",
    "Temperature_Anomaly", "Rainfall_Anomaly"
]

lag_lookup = data_fe[KEY + history_export_cols].copy()

for split_name in ["train", "val", "test"]:
    base_file = PROCESSED_DIR / f"{split_name}.csv"
    if base_file.exists():
        base_df = pd.read_csv(base_file)
        # Avoid duplicate history columns if re-running
        base_df = base_df.drop(columns=[c for c in history_export_cols if c in base_df.columns])
        merged = base_df.merge(lag_lookup, on=KEY, how="left")
    else:
        if split_name == "train":
            merged = data_fe[data_fe["Year"] <= 2011].copy()
        elif split_name == "val":
            merged = data_fe[data_fe["Year"].between(2012, 2014)].copy()
        else:
            merged = data_fe[data_fe["Year"] >= 2015].copy()

    # Save to data/processed/ (for direct handoff to Notebook 04 and model notebooks)
    out_main = PROCESSED_DIR / f"{split_name}_engineered.csv"
    merged.to_csv(out_main, index=False)
    
    # Save to data/processed/feature_engineering/ (modular archive)
    out_archive = OUTPUT_DIR / f"{split_name}_historical_features.csv"
    merged.to_csv(out_archive, index=False)
    
    print(f"Exported {split_name}_engineered.csv: {merged.shape[0]:,} rows, {merged.shape[1]} columns")

# Export feature configurations
with open(PROCESSED_DIR / "feature_sets.json", "w") as f:
    json.dump(feature_sets, f, indent=2)
with open(OUTPUT_DIR / "feature_sets.json", "w") as f:
    json.dump(feature_sets, f, indent=2)

print("\nFeature engineering handoff completed successfully.")


Exported train_engineered.csv: 176,194 rows, 30 columns
Exported val_engineered.csv: 11,345 rows, 30 columns
Exported test_engineered.csv: 10,828 rows, 30 columns

Feature engineering handoff completed successfully.
